<div class="alert alert-block alert-success">
<h1> Machine Learning - Project Cars4You</h1>
<h2> Open Ended Section <h2>
<h2> Group 36 </h2>
<h3> 2025/2026 </h3>

***

## **Table of Contents**<br>

[1. **Importing libraries & Data**](#1-Importing-libraries-&-Data)<br>

[2. Train-test Split](#2-Train-test-Split)<br>

[3. Feature Engineering](#3-Feature-Engineeering)<br>
- [3.1 Encoding](#31-Encoding)<br>
- [3.2 Missing Values](#32-Missing-Values)<br>
    - [3.2.1 Dealing with Missing Values](#221-Dealing-with-Missing-Values)<br>
- [3.3 Outliers](#33-Outliers)<br>
    - [3.3.1 Outliers detection](#331-Outliers-detection)<br>
    - [3.3.2 Dealing with outliers](#332-Dealing-with-outliers)<br>
- [3.4 Scaling](#34-Scaling)<br>
    
[4 Feature Selection](#4-Feature-Selection)<br>

[5 Export](#5-Export)<br>
    

***

This notebook will consist of building an analytics interface to predict car price based on a user input

> Student Name - Artem Polikarpov
>> 20250443

> Student Name - Diogo Montenegro
>> 20250491

> Student Name - Francisco Martins
>> 20250482

> Student Name - João Cardoso
>> 20240529 


***

# 1. Import Libraries & Data

Import core libraries + sklearn utilities.

In [1]:
# sklearn==1.5.1
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn import set_config


from sklearn.model_selection import train_test_split

# Keep pandas output through the pipeline (helps debugging / feature names)
set_config(transform_output="pandas")


#### Data Loading

Load **pre-cleaned** train and test sets coming from the EDA/preprocessing notebook.

In [2]:
train = pd.read_csv('../project_data/train_data_EDA.csv', index_col = 0)

test = pd.read_csv('../project_data/test_data_EDA.csv', index_col = 0)

# 2. Train-Test Split

**Holdout Method** — we create a fixed **validation set** to compare feature sets and models quickly. The same from notebook 2.

In [3]:
# Split the DataFrame into features (X) and target variable (y)
X = train.drop('price', axis=1) 
y = train['price']  
# Holdout split: 80% train / 20% validation
X_train, X_val, y_train, y_val = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=0) 

# 3. Pipeline of the preprocess/feature selection from notebook 2 + the final chosen model.

In [4]:
class ModelTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Target-encode `model` into `model_te` using TRAIN y only, then drop `model`.
    """
    def __init__(self, model_col="model", out_col="model_te", drop_model=True):
        self.model_col = model_col
        self.out_col = out_col
        self.drop_model = drop_model

    def fit(self, X, y):
        if y is None:
            raise ValueError("ModelTargetEncoder requires y in fit().")
        X = X.copy()
        s = pd.Series(y, index=X.index)

        mean_by_model = s.groupby(X[self.model_col]).mean()
        self.mean_by_model_ = mean_by_model
        self.global_mean_ = float(s.mean())
        return self

    def transform(self, X):
        X = X.copy()
        X[self.out_col] = X[self.model_col].map(self.mean_by_model_).astype(float)
        X[self.out_col] = X[self.out_col].fillna(self.global_mean_)
        if self.drop_model and self.model_col in X.columns:
            X = X.drop(columns=[self.model_col])
        return X


class TaxImputeBinOHE(BaseEstimator, TransformerMixin):
    """
    1) impute tax by median tax per model_te (learned on TRAIN), fallback global median
    2) bin tax with fixed bins/labels
    3) OHE tax_bin, then drop raw tax and add tax_bin_* dummies
    """
    def __init__(
        self,
        tax_col="tax",
        key_col="model_te",
        bins=(-0.1, 0.1, 100, 170, 450, np.inf),
        labels=("0", "1_100", "101_170", "171_450", "451_plus"),
    ):
        self.tax_col = tax_col
        self.key_col = key_col
        self.bins = list(bins)
        self.labels = list(labels)

    def fit(self, X, y=None):
        X = X.copy()
        tax = pd.to_numeric(X[self.tax_col], errors="coerce")
        key = pd.to_numeric(X[self.key_col], errors="coerce")

        self.tax_median_by_key_ = tax.groupby(key).median()
        self.global_tax_median_ = float(tax.median())

        tax_bin = pd.cut(tax.fillna(self.global_tax_median_), bins=self.bins, labels=self.labels)
        self.ohe_ = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.ohe_.fit(tax_bin.to_frame(name="tax_bin"))
        self.tax_ohe_cols_ = list(self.ohe_.get_feature_names_out(["tax_bin"]))
        return self

    def transform(self, X):
        X = X.copy()
        tax = pd.to_numeric(X[self.tax_col], errors="coerce")

        key = pd.to_numeric(X[self.key_col], errors="coerce")
        tax = tax.fillna(key.map(self.tax_median_by_key_))
        tax = tax.fillna(self.global_tax_median_)

        tax_bin = pd.cut(tax, bins=self.bins, labels=self.labels)

        tax_ohe = self.ohe_.transform(tax_bin.to_frame(name="tax_bin"))
        tax_ohe_df = pd.DataFrame(tax_ohe, columns=self.tax_ohe_cols_, index=X.index)

        X = X.drop(columns=[c for c in [self.tax_col] if c in X.columns])
        X = pd.concat([X, tax_ohe_df], axis=1)
        return X


# --- selected features ---
SELECTED_FEATURES = [
    'model_te',
    'engineSize',
    'Brand_opel',
    'mpg',
    'transmission_manual',
    'mileage',
    'Brand_audi',
    'fuelType_petrol',
    'carAge',
    'Brand_volkswagen',
    'Brand_mercedes-benz',
    'Brand_hyundai',
    'Brand_bmw',
    'tax_bin_101_170',
    'tax_bin_171_450',
    'tax_bin_1_100',
    'transmission_semi-auto',
    'transmission_automatic',
    'fuelType_diesel',
    'fuelType_hybrid'
]

class FeatureSelector(BaseEstimator, TransformerMixin):
    """
    Selects a fixed list of columns by name.
    If selected columns are missing at transform-time, creates them filled with 0.0.
    """
    def __init__(self, cols):
        self.cols = list(cols)

    def fit(self, X, y=None):
        self.cols_ = list(self.cols)
        return self

    def transform(self, X):
        X = X.copy()
        missing = [c for c in self.cols_ if c not in X.columns]
        for c in missing:
            X[c] = 0.0
        return X.loc[:, self.cols_]


NUM_COLS = ["mileage", "engineSize", "mpg", "carAge", "previousOwners", "model_te"]


encode_scale = ColumnTransformer(
    transformers=[
        ("num_scale", RobustScaler(), NUM_COLS),
    ],
    remainder="passthrough",                 # keeps the tax_bin_* dummy cols
    verbose_feature_names_out=False,
).set_output(transform="pandas")             # CRITICAL: keep pandas, not numpy


preprocess = Pipeline(steps=[
    ("model_te", ModelTargetEncoder(model_col="model", out_col="model_te", drop_model=True)),
    ("tax_bin_ohe", TaxImputeBinOHE(
        tax_col="tax",
        key_col="model_te",
        bins=(-0.1, 0.1, 100, 170, 450, np.inf),
        labels=("0", "1_100", "101_170", "171_450", "451_plus"),
    )),
    ("encode_scale", encode_scale),

    # CRITICAL: use sklearn KNNImputer and keep pandas output
    ("knn_impute", KNNImputer(n_neighbors=8, weights="uniform").set_output(transform="pandas")),
])


base_estimators = [
    ("hgb", HistGradientBoostingRegressor(
        learning_rate=0.06,
        max_iter=900,
        min_samples_leaf=17,
        max_leaf_nodes=80,
        max_bins=255,
        l2_regularization=0.1,
        early_stopping=False,
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=42,
    )),
    ("gb", GradientBoostingRegressor(
        learning_rate=0.1,
        n_estimators=300,
        min_samples_leaf=15,
        min_samples_split=20,
        max_depth=10,
        max_features=0.6,
        subsample=0.8,
        random_state=42,
    )),
    ("hgb2", HistGradientBoostingRegressor(
        learning_rate=0.07,
        max_iter=1100,
        min_samples_leaf=17,
        max_leaf_nodes=50,
        max_bins=255,
        l2_regularization=0.0,
        early_stopping=False,
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=42,
    )),
]

meta_model = LinearRegression()

stack = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_model,
    passthrough=False,
    n_jobs=-1,

)


model = TransformedTargetRegressor(
    regressor=stack,
    func=np.log1p,
    inverse_func=np.expm1,
)

full_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("select_features", FeatureSelector(SELECTED_FEATURES)),
    ("model", model),
])


## 3.1 Fit full_pipe and create "car_price_pipeline.joblib"

In [5]:
# X_train, y_train: pandas objects (y_train as original price, NOT logged)
# X_val optional for evaluation, X_test for submission
#
full_pipe.fit(X_train, y_train)
y_pred = full_pipe.predict(X_val)
test_pred = full_pipe.predict(test)

import joblib

joblib.dump(
    {
        "pipeline": full_pipe,
        "feature_names": list(X_train.columns),
        # Optional but useful for UI dropdowns:
        "train_uniques": {c: sorted(X_train[c].dropna().astype(str).unique().tolist())
                          for c in X_train.select_dtypes(include=["object", "category"]).columns}
    },
    "car_price_pipeline.joblib"
)

C:\Users\artem\anaconda3\envs\ml-project_v1\Lib\site-packages\sklearn\preprocessing\_function_transformer.py:303: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
C:\Users\artem\anaconda3\envs\ml-project_v1\Lib\site-packages\sklearn\preprocessing\_function_transformer.py:303: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
C:\Users\artem\anaconda3\envs\ml-project_v1\Lib\site-packages\sklearn\preprocessing\_function_transformer.py:303: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))
C:\U

['car_price_pipeline.joblib']

In [6]:
!pip install streamlit joblib pandas scikit-learn

   ---------------------------------------- 0.0/9.0 MB ? eta -:--:--
   ------------ --------------------------- 2.9/9.0 MB 15.3 MB/s eta 0:00:01
   ------------------------- -------------- 5.8/9.0 MB 14.1 MB/s eta 0:00:01
   -------------------------------------- - 8.7/9.0 MB 13.8 MB/s eta 0:00:01
   ---------------------------------------- 9.0/9.0 MB 12.8 MB/s  0:00:00
   ---------------------------------------- 0.0/795.4 kB ? eta -:--:--
   ---------------------------------------- 795.4/795.4 kB 11.3 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ------------------ --------------------- 3.1/6.9 MB 15.3 MB/s eta 0:00:01
   ---------------------------------- ----- 6.0/6.9 MB 14.2 MB/s eta 0:00:01
   ---------------------------------------- 6.9/6.9 MB 12.9 MB/s  0:00:00
   ---------------------------------------- 0.0/28.0 MB ? eta -:--:--
   ---- ----------------------------------- 2.9/28.0 MB 15.3 MB/s eta 0:00:02
   ------- ---------------------

In [7]:
import joblib
import pandas as pd
import streamlit as st

ASSET_PATH = "car_price_pipeline.joblib"

@st.cache_resource
def load_asset(path: str):
    return joblib.load(path)

asset = load_asset(ASSET_PATH)
pipe = asset["pipeline"]
feature_names = asset["feature_names"]
train_uniques = asset.get("train_uniques", {})

st.title("Car Price Prediction")
st.write("Enter vehicle data and get a predicted price.")

# Build a form for user input
with st.form("predict_form"):
    inputs = {}

    for col in feature_names:
        # Heuristic: if we have known categories for this column, render a dropdown
        if col in train_uniques and len(train_uniques[col]) > 0:
            options = train_uniques[col]
            # Add a safe fallback option for unseen categories
            options = options + ["__OTHER__"] if "__OTHER__" not in options else options
            inputs[col] = st.selectbox(col, options=options)
            if inputs[col] == "__OTHER__":
                inputs[col] = st.text_input(f"{col} (custom value)", value="")
        else:
            # Numeric input fallback (Streamlit returns float/int depending on step)
            inputs[col] = st.number_input(col, value=0.0)

    submitted = st.form_submit_button("Predict")

if submitted:
    # Create a 1-row dataframe with the exact column order used in training
    x_new = pd.DataFrame([[inputs[c] for c in feature_names]], columns=feature_names)

    # Basic hygiene: empty strings to NA (optional)
    x_new = x_new.replace({"": pd.NA})

    pred = pipe.predict(x_new)[0]

    st.subheader("Prediction")
    st.metric("Predicted price", f"€ {pred:,.0f}")
    st.caption("Note: This is a point estimate (not a guaranteed sale price).")


2025-12-21 00:02:20.432 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-21 00:02:21.170 
  command:

    streamlit run C:\Users\artem\anaconda3\envs\ml-project_v1\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-12-21 00:02:21.170 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-21 00:02:21.186 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-21 00:02:21.186 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-21 00:02:21.698 Thread 'Thread-7': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-21 00:02:21.698 Thread 'Thread-7': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-21 00:02:21.698 Thread '

In [ ]:
!streamlit run app.py --server.address 127.0.0.1 --server.port 8501 --server.headless true